In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


spark = SparkSession.builder.appName("MostObscureMarvelHero").getOrCreate()

# Build hero -> connection count from graph lines.
lines = spark.read.text("MarvelGraph.txt")
connections = (
    lines.select(
        F.split(F.trim(F.col("value")), r"\s+").alias("parts")
    )
    .select(
        F.col("parts")[0].cast("int").alias("id"),
        (F.size(F.col("parts")) - 1).alias("connections")
    )
    .groupBy("id")
    .agg(F.sum("connections").alias("connections"))
)

# Parse names using csv options you used before: id "name"
name_schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("name", StringType(), nullable=False),
])

names = (
    spark.read
    .schema(name_schema)
    .option("sep", " ")
    .option("quote", '"')
    .csv("MarvelNames.txt")
)

# Keep only the minimum connection count (ties included).
min_connections = connections.agg(F.min("connections")).first()[0]
most_obscure = (
    connections
    .filter(F.col("connections") == min_connections)
    .join(names, "id")
    .select("id", "name", "connections")
)

print("Most obscure heroes (ties allowed):")
most_obscure.show(truncate=False)

: 